# E-Commerce Transactions Analysis
**Dataset**: smayanj/e-commerce-transactions-dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('shardul/ecommerce-transactions/ecommerce_transactions.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()


## 1. Data Quality & Schema

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Null Counts ===')
print(df.isnull().sum())
print(f'\n=== Unique Values ===')
for col in df.columns:
    print(f'{col}: {df[col].nunique():,}')


## 2. Statistical Summary

In [ ]:
df.describe()


## 3. Categorical Distributions

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
n = len(cat_cols)
if n > 0:
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 5*((n+1)//2)))
    axes = axes.flatten() if n > 2 else [axes] if n == 1 else axes.flatten()
    for i, col in enumerate(cat_cols):
        if df[col].nunique() <= 30:
            df[col].value_counts().plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
            axes[i].set_title(f'{col} Distribution')
            axes[i].tick_params(axis='x', rotation=45)
        else:
            df[col].value_counts().head(15).plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
            axes[i].set_title(f'{col} (Top 15)')
            axes[i].tick_params(axis='x', rotation=45)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 4. Temporal Analysis

In [ ]:
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f'{col}: {df[col].min()} to {df[col].max()} ({(df[col].max()-df[col].min()).days} days)')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    df[col].dt.dayofweek.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title(f'{col} - Day of Week')
    axes[0].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
    
    df[col].dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title(f'{col} - Monthly Distribution')
    plt.tight_layout()
    plt.show()


## 5. Transaction Value Analysis

In [ ]:
price_cols = [c for c in df.columns if any(k in c.lower() for k in ['price','amount','total','revenue'])]
qty_cols = [c for c in df.columns if any(k in c.lower() for k in ['quantity','qty'])]
value_cols = price_cols + qty_cols

if value_cols:
    fig, axes = plt.subplots(1, len(value_cols), figsize=(7*len(value_cols), 5))
    if len(value_cols) == 1:
        axes = [axes]
    for i, col in enumerate(value_cols):
        if df[col].dtype in ['float64','int64']:
            axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
            axes[i].set_title(f'{col} Distribution')
            axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
            axes[i].legend()
    plt.tight_layout()
    plt.show()

for col in price_cols:
    if df[col].dtype in ['float64','int64']:
        print(f'{col} — Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f}, Total: {df[col].sum():,.2f}')


## 6. Customer Analysis

In [ ]:
cust_cols = [c for c in df.columns if any(k in c.lower() for k in ['customer','user','client'])]
if cust_cols:
    cc = cust_cols[0]
    print(f'Unique customers: {df[cc].nunique():,}')
    
    cust_txn = df.groupby(cc).size()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(cust_txn.values, bins=50, edgecolor='black', alpha=0.7, color='teal')
    ax.set_title('Transactions per Customer')
    ax.set_xlabel('Number of Transactions')
    ax.axvline(cust_txn.mean(), color='red', linestyle='--', label=f'Mean: {cust_txn.mean():.1f}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    if price_cols:
        cust_spend = df.groupby(cc)[price_cols[0]].sum()
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.hist(cust_spend.values, bins=50, edgecolor='black', alpha=0.7, color='salmon')
        ax.set_title(f'Total {price_cols[0]} per Customer')
        ax.axvline(cust_spend.mean(), color='red', linestyle='--', label=f'Mean: {cust_spend.mean():.2f}')
        ax.legend()
        plt.tight_layout()
        plt.show()


## 7. Product Analysis

In [ ]:
prod_cols = [c for c in df.columns if any(k in c.lower() for k in ['product','item','sku'])]
cat_cols_prod = [c for c in df.columns if 'category' in c.lower()]

if prod_cols:
    pc = prod_cols[0]
    print(f'Unique products: {df[pc].nunique():,}')
    
    top = df[pc].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(14, 8))
    top.plot(kind='barh', ax=ax, color=sns.color_palette('magma', 20))
    ax.set_title(f'Top 20 Products by Frequency')
    ax.set_xlabel('Count')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

if cat_cols_prod and price_cols:
    for col in cat_cols_prod:
        fig, ax = plt.subplots(figsize=(14, 6))
        cat_rev = df.groupby(col)[price_cols[0]].sum().sort_values(ascending=True)
        cat_rev.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(cat_rev)))
        ax.set_title(f'Revenue by {col}')
        ax.set_xlabel(f'Total {price_cols[0]}')
        plt.tight_layout()
        plt.show()


## 8. Relevance to Shelf Optimization / Planogram AI

**Strengths:**
- Revenue/price data enables margin-based SKU scoring
- Product velocity from transaction frequency
- Customer basket analysis for co-purchase detection
- Category performance for shelf allocation

**Limitations:**
- E-commerce data — no physical shelf/aisle information
- May not directly map to in-store shopping patterns
- No physical constraint data (shelf dimensions, facings)
